# Testy czarnoskrzynkowe — Mealie

## Cel
Sprawdzić, czy wejścia oraz wyjścia wybranych funkcji są zgodne z ich założeniami - testy czarnoskrzynkowe.

## Metoda
Dla każdej funkcji tworzę scenariusze testowe z podziałem na:

| Typ | Co sprawdza |
|---|---|
| **poprawne** | typowe, prawidłowe dane wejściowe |
| **brzegowe** | wartości graniczne (pusty tekst, przełom roku, dokładnie `max_length`, …) |
| **niepoprawne** | dane, z którymi funkcja powinna sobie poradzić lub zgłosić błąd |

Każdy przypadek ma **ID** (`KS-xx`), opis, wejście i oczekiwany wynik. Wynik to `PASS` / `FAIL` — przy `FAIL` notebook
pokazuje, co funkcja zwróciła zamiast oczekiwanej wartości.

## Testowane funkcje

| # | Funkcja | Rola w systemie | Przypadki |
|---|---|---|---|
| 1 | `create_recipe_slug(name, max_length=250)` | adres URL przepisu tworzony z nazwy | KS-01 … KS-10 |
| 2 | `QueryFilterBuilder.combine_filters(*filter_strings)` | łączenie filtrów wyszukiwania | KS-11 … KS-19 |
| 3 | `PlanRulesDay.from_date(date)` | dzień tygodnia posiłku w planerze | KS-20 … KS-28 |

> Pozostałe dwie funkcje z przydziału (`Hasher.verify`, `create_access_token`) nie są jeszcze zaimplementowane.

## Przygotowanie środowiska
Importy, narzędzia do uruchamiania przypadków i wyświetlania wyników w tabeli.

In [ ]:
import datetime
import locale
import re
import sys
from dataclasses import dataclass
from html import escape
from pathlib import Path
from typing import Any, Callable

from IPython.display import HTML, display

# Notebook można uruchomić z dowolnego katalogu wewnątrz repozytorium
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mealie").is_dir())
sys.path.insert(0, str(ROOT))

from fastapi import HTTPException

from mealie.schema.meal_plan.plan_rules import PlanRulesDay
from mealie.schema.recipe.recipe import create_recipe_slug
from mealie.services.query_filter.builder import QueryFilterBuilder

print("Repozytorium:", ROOT)

In [ ]:
class Skip(Exception):
    """Przypadek nie może być wykonany na tej maszynie."""


@dataclass
class Case:
    id: str
    desc: str
    kind: str                  # poprawne / brzegowe / niepoprawne
    inp: str                   # wejście (do wyświetlenia)
    expected: str              # oczekiwany wynik (do wyświetlenia)
    check: Callable[[], None]  # rzuca AssertionError, gdy funkcja zachowuje się inaczej


RESULTS: dict[str, list[dict]] = {}   # nazwa funkcji -> wyniki (do podsumowania na końcu)
COLORS = {"PASS": "#1a7f37", "FAIL": "#cf222e", "SKIP": "#9a6700"}


def expect(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)


def expect_raises(exc: type[BaseException] | tuple[type[BaseException], ...], fn: Callable[[], Any]) -> BaseException:
    """Wywołuje fn i sprawdza, że zgłasza wyjątek podanego typu — opisuje, co poszło inaczej."""
    names = " / ".join(e.__name__ for e in (exc if isinstance(exc, tuple) else (exc,)))
    try:
        result = fn()
    except exc as e:
        return e
    except Exception as e:
        raise AssertionError(f"rzuca {type(e).__name__} zamiast {names}") from None
    raise AssertionError(f"brak wyjątku, zwrócono {result!r} (oczekiwano {names})")


def run_cases(function_name: str, cases: list[Case]) -> None:
    rows = []
    for c in cases:
        try:
            c.check()
            status, note = "PASS", ""
        except Skip as e:
            status, note = "SKIP", str(e)
        except AssertionError as e:
            status, note = "FAIL", str(e)
        except Exception as e:  # nieoczekiwany błąd w samym teście / funkcji
            status, note = "FAIL", f"{type(e).__name__}: {e}"
        rows.append({"id": c.id, "desc": c.desc, "kind": c.kind, "inp": c.inp,
                     "expected": c.expected, "status": status, "note": note})
    RESULTS[function_name] = rows
    show_table(rows)


def show_table(rows: list[dict]) -> None:
    head = ["ID", "Opis", "Typ", "Wejście", "Oczekiwane", "Wynik"]
    html = ["<table style='border-collapse:collapse;font-size:13px'>",
            "<tr>" + "".join(f"<th style='border:1px solid #8884;padding:4px 8px;text-align:left'>{h}</th>" for h in head) + "</tr>"]
    for r in rows:
        color = COLORS[r["status"]]
        result = f"<b style='color:{color}'>{r['status']}</b>"
        if r["note"]:
            result += f"<br><span style='font-size:12px'>{escape(r['note'])}</span>"
        cells = [escape(r["id"]), escape(r["desc"]), escape(r["kind"]),
                 f"<code>{escape(r['inp'])}</code>", f"<code>{escape(r['expected'])}</code>", result]
        html.append("<tr>" + "".join(f"<td style='border:1px solid #8884;padding:4px 8px;vertical-align:top'>{c}</td>" for c in cells) + "</tr>")
    html.append("</table>")
    passed = sum(r["status"] == "PASS" for r in rows)
    failed = sum(r["status"] == "FAIL" for r in rows)
    html.append(f"<p><b>{len(rows)} przypadków:</b> "
                f"<span style='color:{COLORS['PASS']}'>{passed} PASS</span>, "
                f"<span style='color:{COLORS['FAIL']}'>{failed} FAIL</span></p>")
    display(HTML("".join(html)))

---
## 1. `create_recipe_slug` — adres URL przepisu

```python
def create_recipe_slug(name: str, max_length: int = 250) -> str
```

**Kontrakt z docstringa**
- zamienia nazwę przepisu na *slug* (adres URL),
- wynik jest krótszy lub równy `max_length`,
- dla nazwy, której nie da się zamienić na slug → `ValueError`.

**Założenie:** poprawny slug to małe litery i cyfry rozdzielone pojedynczymi myślnikami, **bez myślnika na początku
i końcu**. Obcięcie do `max_length` nie może zostawić „wiszącego" myślnika.

In [ ]:
SLUG_PATTERN = re.compile(r"^[a-z0-9]+(-[a-z0-9]+)*$")


def slug_equals(name: str, expected: str) -> Callable[[], None]:
    def check():
        slug = create_recipe_slug(name)
        expect(slug == expected, f"zwraca {slug!r}")
        expect(bool(SLUG_PATTERN.match(slug)), f"niepoprawny format sluga: {slug!r}")
    return check


def slug_truncated(name: str, max_length: int) -> Callable[[], None]:
    def check():
        slug = create_recipe_slug(name, max_length=max_length)
        expect(0 < len(slug) <= max_length, f"długość {len(slug)} przy max_length={max_length}")
        expect(bool(SLUG_PATTERN.match(slug)), f"zwraca {slug!r} (myślnik na końcu adresu)")
        expect(create_recipe_slug(name, max_length=10_000).startswith(slug), "to nie jest początek pełnego sluga")
    return check


def slug_exact_length() -> None:
    name = "b" * 250
    expect(create_recipe_slug(name) == name, "slug o długości dokładnie max_length został zmieniony")


def slug_raises_value_error(name: str) -> Callable[[], None]:
    return lambda: expect_raises(ValueError, lambda: create_recipe_slug(name))


slug_cases = [
    Case("KS-01", "Zwykła nazwa",             "poprawne",    '"Pasta Fagioli"',              '"pasta-fagioli"',        slug_equals("Pasta Fagioli", "pasta-fagioli")),
    Case("KS-02", "Wielkie litery",           "poprawne",    '"SPAGHETTI Carbonara"',        '"spaghetti-carbonara"',  slug_equals("SPAGHETTI Carbonara", "spaghetti-carbonara")),
    Case("KS-03", "Nadmiarowe spacje",        "brzegowe",    '"  Pasta    Fagioli  "',       '"pasta-fagioli"',        slug_equals("  Pasta    Fagioli  ", "pasta-fagioli")),
    Case("KS-04", "Polskie znaki",            "niepoprawne", '"Żurek łódzki z jajkiem"',     '"zurek-lodzki-z-jajkiem"', slug_equals("Żurek łódzki z jajkiem", "zurek-lodzki-z-jajkiem")),
    Case("KS-05", "Znaki specjalne",          "niepoprawne", '"Chili!!! @ Home #1"',         '"chili-home-1"',         slug_equals("Chili!!! @ Home #1", "chili-home-1")),
    Case("KS-06", "Nazwa dłuższa niż domyślne max_length", "brzegowe", '"a" * 300',          "długość ≤ 250, poprawny slug", slug_truncated("a" * 300, 250)),
    Case("KS-07", "Własne max_length ucinające w środku nazwy", "brzegowe", '"Spaghetti Carbonara z boczkiem", max_length=10', "długość ≤ 10, bez myślnika na końcu", slug_truncated("Spaghetti Carbonara z boczkiem", 10)),
    Case("KS-08", "Slug o długości dokładnie max_length", "brzegowe", '"b" * 250',            '"b" * 250 (bez ucinania)', slug_exact_length),
    Case("KS-09", "Pusta nazwa",              "brzegowe",    '""',                           "ValueError (wg docstringa)", slug_raises_value_error("")),
    Case("KS-10", "Same znaki specjalne",     "niepoprawne", '"!!! ??? ###"',                "ValueError (wg docstringa)", slug_raises_value_error("!!! ??? ###")),
]

run_cases("create_recipe_slug", slug_cases)

### Podgląd na żywo


In [ ]:
print("KS-04 :", create_recipe_slug("Żurek łódzki z jajkiem"))
print("KS-07 :", repr(create_recipe_slug("Spaghetti Carbonara z boczkiem", max_length=10)), " <- myślnik na końcu")
try:
    create_recipe_slug("")
except Exception as e:
    print("KS-09 :", type(e).__name__, "| dziedziczy po ValueError?", isinstance(e, ValueError))

---
## 2. `QueryFilterBuilder.combine_filters` — łączenie filtrów

```python
QueryFilterBuilder.combine_filters(*filter_strings: str | None) -> str
```

**Kontrakt (z nazwy i sygnatury)**
- łączy filtry w jeden, każdy kolejny dołączany operatorem `AND`, z zachowaniem kolejności,
- `None` i `""` są pomijane; brak filtrów → `""`,
- filtr z `OR` łączony z innym musi zostać **zgrupowany**: `(a OR b) AND c`, bo inaczej `a OR b AND c`
  oznaczałoby `a OR (b AND c)`,
- niepoprawna składnia → błąd HTTP 4xx (`HTTPException`), a nie awaria serwera (5xx).

In [ ]:
FILTER_NAME = 'name = "Pasta Fagioli"'
FILTER_DATE = 'createdAt >= "2021-02-22"'
FILTER_TAGS = 'tags.name CONTAINS ALL ["Easy", "Cajun"]'
FILTER_OR = 'name = "Pasta Fagioli" OR name = "Grandma\'s Brisket"'


def combine_single() -> None:
    result = QueryFilterBuilder.combine_filters(FILTER_NAME)
    expect(FILTER_NAME in result, f"filtr nie występuje w wyniku: {result!r}")
    expect("AND" not in result, f"niepotrzebny AND: {result!r}")


def combine_and(*filters: str) -> Callable[[], None]:
    def check():
        result = QueryFilterBuilder.combine_filters(*filters)
        expect(all(f in result for f in filters), f"brakuje filtra w wyniku: {result!r}")
        expect(result.count(" AND ") == len(filters) - 1, f"liczba AND = {result.count(' AND ')}: {result!r}")
        positions = [result.index(f) for f in filters]
        expect(positions == sorted(positions), f"zmieniona kolejność: {result!r}")
    return check


def combine_empty(*filters: str | None) -> Callable[[], None]:
    def check():
        result = QueryFilterBuilder.combine_filters(*filters)
        expect(result == "", f"zwraca {result!r} zamiast pustego napisu")
    return check


def combine_ignores_empty() -> None:
    with_empty = QueryFilterBuilder.combine_filters(None, FILTER_NAME, "", FILTER_DATE, None)
    without_empty = QueryFilterBuilder.combine_filters(FILTER_NAME, FILTER_DATE)
    expect(with_empty == without_empty, f"{with_empty!r} != {without_empty!r}")


def combine_keeps_or_grouped() -> None:
    result = QueryFilterBuilder.combine_filters(FILTER_OR, FILTER_DATE)
    expect(f"({FILTER_OR})" in result, f"OR niezgrupowany: {result!r}")
    expect(" AND " in result, f"brak AND: {result!r}")


def combine_invalid(*filters: str) -> Callable[[], None]:
    def check():
        e = expect_raises(HTTPException, lambda: QueryFilterBuilder.combine_filters(*filters))
        expect(400 <= e.status_code < 500, f"kod {e.status_code} zamiast 4xx")
    return check


filter_cases = [
    Case("KS-11",  "Jeden filtr",                    "poprawne",    "name",                  "wynik zawiera filtr, brak AND", combine_single),
    Case("KS-12",  "Dwa filtry",                     "poprawne",    "nazwa, data",           "oba filtry, jedno AND, kolejność zachowana", combine_and(FILTER_NAME, FILTER_DATE)),
    Case("KS-13",  "Trzy filtry (w tym CONTAINS ALL)", "poprawne",  "nazwa, data, tagi",     "trzy filtry, dwa AND, kolejność zachowana", combine_and(FILTER_NAME, FILTER_DATE, FILTER_TAGS)),
    Case("KS-14",  "Brak argumentów",                "brzegowe",    "()",                    '""', combine_empty()),
    Case("KS-15",  "Same None",                      "brzegowe",    "None, None, None",      '""', combine_empty(None, None, None)),
    Case("KS-16",  "Same puste napisy",              "brzegowe",    '"", None, ""',          '""', combine_empty("", None, "")),
    Case("KS-17",  "Puste części pomiędzy filtrami", "brzegowe",    'None, f1, "", f2, None', "to samo co combine_filters(f1, f2)", combine_ignores_empty),
    Case("KS-18",  "Filtr z OR łączony z innym",     "poprawne",    "'a OR b', c",           "(a OR b) zgrupowane w nawiasie + AND", combine_keeps_or_grouped),
    Case("KS-19a", "Brak wartości",                  "niepoprawne", "'name = \"Pasta\"', 'name ='",  "HTTPException 4xx", combine_invalid('name = "Pasta"', "name =")),
    Case("KS-19b", "Niezamknięty nawias",            "niepoprawne", "'(name = \"Pasta\"'",   "HTTPException 4xx", combine_invalid('(name = "Pasta"')),
    Case("KS-19c", "Brak operatora",                 "niepoprawne", "'name = \"Pasta\"', 'name \"Pasta\"'", "HTTPException 4xx", combine_invalid('name = "Pasta"', 'name "Pasta"')),
]

run_cases("QueryFilterBuilder.combine_filters", filter_cases)

### Podgląd na żywo

In [ ]:
print("KS-12 :", QueryFilterBuilder.combine_filters(FILTER_NAME, FILTER_DATE))
print("KS-18 :", QueryFilterBuilder.combine_filters(FILTER_OR, FILTER_DATE))
print("KS-19c:", QueryFilterBuilder.combine_filters('name = "Pasta"', 'name "Pasta"'), " <- brak błędu mimo złej składni")

---
## 3. `PlanRulesDay.from_date` — dzień tygodnia w planerze

```python
PlanRulesDay.from_date(date: datetime.date)
```

**Kontrakt (z nazwy i docstringa)** — *„Returns the enum value for the date passed in"*
- zwraca dzień tygodnia (`monday` … `sunday`) dla podanej daty,
- wynik **nie może zależeć** od języka systemu na serwerze,
- `unset` jest wartością „brak dnia" — dla prawdziwej daty nie powinien się pojawić,
- zły typ wejścia (`None`, napis) → `TypeError` / `AttributeError`.

In [ ]:
D = PlanRulesDay


def day_is(date: datetime.date, expected: PlanRulesDay) -> Callable[[], None]:
    def check():
        result = PlanRulesDay.from_date(date)
        expect(result == expected, f"zwraca {result!r}")
    return check


def week_is_distinct() -> None:
    start = datetime.date(2026, 12, 28)   # tydzień przechodzący przez nowy rok
    days = [PlanRulesDay.from_date(start + datetime.timedelta(days=i)) for i in range(7)]
    expect(len(set(days)) == 7, f"powtórzone dni: {[d.value for d in days]}")
    expect(PlanRulesDay.unset not in days, f"w wyniku jest unset: {[d.value for d in days]}")


def invalid_input(value: Any) -> Callable[[], None]:
    return lambda: expect_raises((TypeError, AttributeError), lambda: PlanRulesDay.from_date(value))


def independent_of_locale() -> None:
    names = ("pl_PL.UTF-8", "pl_PL.utf8", "pl_PL", "Polish_Poland.1250", "Polish")
    original = locale.setlocale(locale.LC_TIME)
    try:
        for name in names:
            try:
                locale.setlocale(locale.LC_TIME, name)
                break
            except locale.Error:
                continue
        else:
            raise Skip("brak polskiego locale na tej maszynie")
        result = PlanRulesDay.from_date(datetime.date(2026, 10, 19))
        weekday_pl = datetime.date(2026, 10, 19).strftime("%A")
    finally:
        locale.setlocale(locale.LC_TIME, original)
    expect(result == D.monday, f"zwraca {result!r} (system nazywa ten dzień {weekday_pl!r})")


date_cases = [
    Case("KS-20",  "Zwykła data",          "poprawne", "date(2026, 10, 19)", "monday",    day_is(datetime.date(2026, 10, 19), D.monday)),
    Case("KS-21",  "Ostatni dzień roku",   "brzegowe", "date(2025, 12, 31)", "wednesday", day_is(datetime.date(2025, 12, 31), D.wednesday)),
    Case("KS-22",  "Pierwszy dzień roku",  "brzegowe", "date(2026, 1, 1)",   "thursday",  day_is(datetime.date(2026, 1, 1), D.thursday)),
    Case("KS-23",  "29 lutego (rok przestępny)", "brzegowe", "date(2024, 2, 29)", "thursday", day_is(datetime.date(2024, 2, 29), D.thursday)),
    Case("KS-24a", "Najmniejsza możliwa data", "brzegowe", "date.min (0001-01-01)", "monday", day_is(datetime.date.min, D.monday)),
    Case("KS-24b", "Największa możliwa data",  "brzegowe", "date.max (9999-12-31)", "friday", day_is(datetime.date.max, D.friday)),
    Case("KS-25",  "datetime późnym wieczorem", "brzegowe", "datetime(2026, 10, 18, 23, 59, 59, UTC)", "sunday",
         day_is(datetime.datetime(2026, 10, 18, 23, 59, 59, tzinfo=datetime.UTC), D.sunday)),
    Case("KS-26",  "Tydzień na przełomie roku", "brzegowe", "7 kolejnych dni od 2026-12-28", "7 różnych dni, żaden nie unset", week_is_distinct),
    Case("KS-27a", "None zamiast daty",     "niepoprawne", "None",         "TypeError / AttributeError", invalid_input(None)),
    Case("KS-27b", "Data jako napis",       "niepoprawne", '"2026-10-19"', "TypeError / AttributeError", invalid_input("2026-10-19")),
    Case("KS-28",  "Serwer z polskim locale (LC_TIME)", "niepoprawne", "date(2026, 10, 19) przy locale pl_PL", "monday", independent_of_locale),
]

run_cases("PlanRulesDay.from_date", date_cases)

### Podgląd na żywo — KS-28

In [ ]:
d = datetime.date(2026, 10, 19)
print("locale domyślne :", PlanRulesDay.from_date(d))
for name in ("pl_PL.UTF-8", "Polish_Poland.1250", "Polish"):
    try:
        locale.setlocale(locale.LC_TIME, name)
    except locale.Error:
        continue
    print(f"locale {name:<20}: strftime('%A') = {d.strftime('%A')!r} -> {PlanRulesDay.from_date(d)!r}")
    break
else:
    print("brak polskiego locale na tej maszynie")
locale.setlocale(locale.LC_TIME, "C")

---
## Podsumowanie

In [ ]:
total = {"PASS": 0, "FAIL": 0, "SKIP": 0}
summary = ["<table style='border-collapse:collapse;font-size:14px'><tr>"
           + "".join(f"<th style='border:1px solid #8884;padding:4px 10px;text-align:left'>{h}</th>" for h in ["Funkcja", "Przypadki", "PASS", "FAIL", "SKIP"])
           + "</tr>"]
for fn, rows in RESULTS.items():
    n = {s: sum(r["status"] == s for r in rows) for s in total}
    for s in total:
        total[s] += n[s]
    summary.append("<tr>" + "".join(f"<td style='border:1px solid #8884;padding:4px 10px'>{c}</td>" for c in
                   [f"<code>{fn}</code>", len(rows), n["PASS"], n["FAIL"], n["SKIP"]]) + "</tr>")
summary.append("<tr style='font-weight:bold'>" + "".join(f"<td style='border:1px solid #8884;padding:4px 10px'>{c}</td>" for c in
               ["Razem", sum(total.values()), total["PASS"], total["FAIL"], total["SKIP"]]) + "</tr></table>")
display(HTML("".join(summary)))

failed = [r for rows in RESULTS.values() for r in rows if r["status"] == "FAIL"]
show_table(failed)

## Wykryte rozbieżności (wnioski)

| ID | Funkcja | Problem | Skutek dla użytkownika |
|---|---|---|---|
| KS-07 | `create_recipe_slug` | po obcięciu do `max_length` slug może kończyć się myślnikiem (`"spaghetti-"`) | brzydki / niepoprawny adres URL przepisu |
| KS-09, KS-10 | `create_recipe_slug` | rzuca `SlugError`, który **nie** dziedziczy po `ValueError` — wbrew docstringowi | kod łapiący `ValueError` nie obsłuży błędu |
| KS-19a | `combine_filters` | `'name ='` kończy się `IndexError` zamiast `HTTPException` 4xx | w API błąd **500** zamiast czytelnego 4xx |
| KS-19c | `combine_filters` | filtr bez operatora (`'name "Pasta"'`) jest przyjmowany bez błędu | błędne zapytanie przechodzi dalej |
| KS-28 | `PlanRulesDay.from_date` | używa `strftime("%A")`, więc wynik zależy od języka systemu; przy polskim locale zwraca `unset` | planer nie rozpoznaje dni na serwerze z polskimi ustawieniami |

**Wniosek:** testy czarnoskrzynkowe oparte wyłącznie na kontrakcie (sygnatura + docstring) wykryły 6 niezgodności
w 3 funkcjach — w tym dwie realne wady (KS-19a → błąd 500, KS-28 → zależność od locale), których nie widać przy
testowaniu tylko „szczęśliwej ścieżki".